we implement the Sugeno fuzzy inference system from scratch without any fuzzy libraries

the main difference from Mamdani is in the defuzzification step instead of using
a fuzzy output set, Sugeno uses a weighted average of crisp output values

Input variables: nkill, nwound, propextent, attack_encoded, weapon_encoded
Output variable: severity_index (0 to 100)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def trimf(x, a, b, c):
    return np.maximum(0, np.minimum((x - a) / (b - a + 1e-9),
                                     (c - x) / (c - b + 1e-9)))

def trapmf(x, a, b, c, d):
    return np.maximum(0, np.minimum(
        np.minimum((x - a) / (b - a + 1e-9), 1),
        (d - x) / (d - c + 1e-9)
    ))

def fuzzify_nkill(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 0, 0, 1, 4)[0]),
        "Medium":  float(trimf(x, 2, 6, 12)[0]),
        "High":    float(trimf(x, 6, 15, 30)[0]),
        "Extreme": float(trapmf(x, 25, 40, 50, 50)[0]),
    }

def fuzzify_nwound(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 0, 0, 2, 6)[0]),
        "Medium":  float(trimf(x, 3, 10, 20)[0]),
        "High":    float(trimf(x, 15, 35, 60)[0]),
        "Extreme": float(trapmf(x, 45, 65, 80, 80)[0]),
    }

def fuzzify_propextent(val):
    x = np.array([val], dtype=float)
    return {
        "None":         float(trapmf(x, 0, 0, 0, 0.5)[0]),
        "Minor":        float(trimf(x, 0.5, 1, 1.5)[0]),
        "Major":        float(trimf(x, 1.5, 2, 2.5)[0]),
        "Catastrophic": float(trapmf(x, 2.5, 3, 3, 3)[0]),
    }

def fuzzify_attack(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 1, 1, 1, 1.8)[0]),
        "Medium":  float(trimf(x, 1.5, 2, 2.5)[0]),
        "High":    float(trimf(x, 2.5, 3, 3.5)[0]),
        "Extreme": float(trapmf(x, 3.2, 3.6, 5, 5)[0]),
    }

def fuzzify_weapon(val):
    x = np.array([val], dtype=float)
    return {
        "Low":     float(trapmf(x, 1, 1, 1, 1.8)[0]),
        "Medium":  float(trimf(x, 1.5, 2, 2.5)[0]),
        "High":    float(trimf(x, 2.5, 3, 3.5)[0]),
        "Extreme": float(trapmf(x, 3.2, 3.6, 5, 5)[0]),
    }

## Rule Base

in Sugeno, each rule has a crisp constant output value instead of a fuzzy set

we assign the midpoint of each severity level as the crisp output

| Severity | Crisp Value |
|----------|-------------|
| Low      | 12.5        |
| Medium   | 37.5        |
| High     | 62.5        |
| Critical | 87.5        |

In [ ]:
crisp_output = {
    "Low":      12.5,
    "Medium":   37.5,
    "High":     62.5,
    "Critical": 87.5,
}

rules = [
    ("Low",    "Low",    "None",         "Low",    "Low",    "Low"),
    ("Low",    "Low",    "None",         "Low",    "Medium", "Low"),
    ("Low",    "Low",    "Minor",        "Low",    "Low",    "Low"),
    ("Low",    "Low",    "Major",        "Low",    "Low",    "Medium"),
    ("Medium", "Low",    "None",         "Low",    "Low",    "Medium"),
    ("Medium", "Medium", "Minor",        "Medium", "Medium", "Medium"),
    ("Low",    "Medium", "Major",        "Low",    "Medium", "Medium"),
    ("Low",    "Low",    "None",         "High",   "High",   "Medium"),
    ("Low",    "Low",    "Minor",        "Medium", "High",   "Medium"),
    ("Low",    "Low",    "None",         "Medium", "Medium", "Medium"),
    ("Low",    "Low",    "None",         "High",   "Medium", "Medium"),
    ("Low",    "Low",    "None",         "Medium", "High",   "Medium"),
    ("Low",    "Low",    "Minor",        "High",   "Medium", "Medium"),
    ("Low",    "Low",    "None",         "Extreme","Medium", "Medium"),
    ("Low",    "Low",    "None",         "Medium", "Extreme","Medium"),
    ("Medium", "Medium", "Major",        "Medium", "High",   "High"),
    ("High",   "Low",    "Minor",        "High",   "Medium", "High"),
    ("High",   "Medium", "None",         "High",   "High",   "High"),
    ("Medium", "High",   "Major",        "Medium", "High",   "High"),
    ("High",   "High",   "Minor",        "High",   "Medium", "High"),
    ("Low",    "High",   "Major",        "Extreme","High",   "High"),
    ("Medium", "Low",    "Major",        "High",   "Extreme","High"),
    ("Low",    "Low",    "None",         "Extreme","Extreme","High"),
    ("Low",    "Low",    "Minor",        "Extreme","High",   "High"),
    ("Medium", "Low",    "None",         "Extreme","High",   "High"),
    ("Low",    "Medium", "None",         "Extreme","Extreme","High"),
    ("High",   "Medium", "Major",        "High",   "Extreme","Critical"),
    ("High",   "High",   "Major",        "Extreme","Extreme","Critical"),
    ("Extreme","High",   "Major",        "Extreme","High",   "Critical"),
    ("Extreme","Extreme","Catastrophic", "Extreme","Extreme","Critical"),
    ("High",   "High",   "Catastrophic", "High",   "Extreme","Critical"),
    ("Extreme","Medium", "Major",        "Extreme","High",   "Critical"),
]

## sugeno inference

for each rule we compute the firing strength using the minimum operator (AND)

the final crisp output is computed using weighted average of all crisp output values

Formula: z* = sum(strength_i * crisp_i) / sum(strength_i)

In [ ]:
def sugeno_infer(nkill_val, nwound_val, prop_val, atk_val, wpn_val):
    fk  = fuzzify_nkill(nkill_val)
    fw  = fuzzify_nwound(nwound_val)
    fp  = fuzzify_propextent(prop_val)
    fa  = fuzzify_attack(atk_val)
    fwp = fuzzify_weapon(wpn_val)

    numerator   = 0.0
    denominator = 0.0

    for (k, w, p, a, wp, out) in rules:
        strength = min(fk[k], fw[w], fp[p], fa[a], fwp[wp])
        numerator   += strength * crisp_output[out]
        denominator += strength

    if denominator == 0:
        return 0.0

    return numerator / denominator

## single inference example

testing on the same input as Mamdani for a direct comparison

In [ ]:
nkill_test  = 20
nwound_test = 10
prop_test   = 2
atk_test    = 3
wpn_test    = 4

result = sugeno_infer(nkill_test, nwound_test, prop_test, atk_test, wpn_test)

print(f"Input  : nkill={nkill_test}, nwound={nwound_test}, propextent={prop_test}, attack={atk_test}, weapon={wpn_test}")
print(f"Output : severity_score = {result:.2f}")

if result < 25:
    label = "Low"
elif result < 50:
    label = "Medium"
elif result < 75:
    label = "High"
else:
    label = "Critical"

print(f"Label  : {label}")

## running on full dataset

In [ ]:
df = pd.read_csv("../data/gtd_processed.csv")

prop_map = {1: 3, 2: 2, 3: 1, 4: 0}
df["prop_inverted"] = df["propextent"].map(prop_map)

print(f"Loaded {len(df):,} rows")
df[["nkill", "nwound", "propextent", "attack_encoded",
    "weapon_encoded", "severity_index"]].head()

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()

def sugeno_predict(row):
    return sugeno_infer(
        row["nkill"], row["nwound"], row["prop_inverted"],
        row["attack_encoded"], row["weapon_encoded"]
    )

df["sugeno_score"] = df.progress_apply(sugeno_predict, axis=1)

def score_to_label(score):
    if score < 25:
        return "Low"
    elif score < 50:
        return "Medium"
    elif score < 75:
        return "High"
    else:
        return "Critical"

df["sugeno_label"] = df["sugeno_score"].apply(score_to_label)

print("Done.")
print(df["sugeno_label"].value_counts())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_true = df["severity_index"]
y_pred = df["sugeno_label"]
order  = ["Low", "Medium", "High", "Critical"]

acc = accuracy_score(y_true, y_pred)
print(f"Sugeno Accuracy: {acc:.4f} ({acc*100:.2f}%)\n")
print(classification_report(y_true, y_pred, labels=order, target_names=order))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

order = ["Low", "Medium", "High", "Critical"]
cm = confusion_matrix(y_true, y_pred, labels=order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=order)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Sugeno Confusion Matrix")
plt.tight_layout()
plt.show()